# RQ2-v3 quick trajectory gradient diagnostic — T4×2

Read-only mechanism probe over Geo-HT and Resource-HT seed-3 checkpoints at epochs 10, 50, and 100. Each GPU handles one complete trajectory. No training, optimizer step, accuracy, validation selection, test access, or policy update is performed.

In [ ]:
import os, subprocess, sys, json, time, zipfile
from pathlib import Path
from kaggle_secrets import UserSecretsClient
github_token = UserSecretsClient().get_secret('github_token')
assert github_token, 'Missing Kaggle secret github_token'
PROJECT_ROOT = Path('/kaggle/working/new-pruning')
askpass = Path('/kaggle/working/.github_git_askpass.py')
askpass.write_text("#!/usr/bin/env python3\nimport os,sys\np=sys.argv[1] if len(sys.argv)>1 else ''\nprint('x-access-token' if 'Username' in p else os.environ['GITHUB_TOKEN_RUNTIME'])\n")
askpass.chmod(0o700)
env = os.environ.copy(); env.update({'GIT_ASKPASS':str(askpass),'GIT_TERMINAL_PROMPT':'0','GITHUB_TOKEN_RUNTIME':github_token})
try:
    command = ['git','-C',str(PROJECT_ROOT),'pull','--ff-only'] if (PROJECT_ROOT/'.git').is_dir() else ['git','clone','https://github.com/duyh80456-code/new-pruning.git',str(PROJECT_ROOT)]
    subprocess.run(command, env=env, check=True)
finally:
    askpass.unlink(missing_ok=True); github_token = None
os.chdir(PROJECT_ROOT); sys.path.insert(0, str(PROJECT_ROOT))
import torch
assert torch.cuda.device_count() == 2, f'Select Kaggle T4 x2; detected {torch.cuda.device_count()} GPU(s)'
GIT_COMMIT = subprocess.run(['git','rev-parse','HEAD'],capture_output=True,text=True,check=True).stdout.strip()
print('Commit:', GIT_COMMIT)
print('GPUs:', [torch.cuda.get_device_name(i) for i in range(2)])

## Resolve the completed Geo-HT/Resource-HT development output

In [ ]:
import importlib
import rq2_quick_trajectory_diagnostic
rq2_quick_trajectory_diagnostic = importlib.reload(rq2_quick_trajectory_diagnostic)
HT_ROOT = rq2_quick_trajectory_diagnostic.find_ht_development_root(
    Path('/kaggle/input'), '/kaggle/working/materialized-rq2-v3-ht-seed3'
)
print('HT development root:', HT_ROOT)
print('Checkpoints:')
for method in ('geo_ht','resource_ht'):
    print(method, [str(HT_ROOT/method/'seed_3'/f'epoch_{epoch:03d}.pt') for epoch in (10,50,100)])

## Run both trajectories concurrently

GPU 0 probes the Geo-HT path and GPU 1 probes the Resource-HT path. Both workers reconstruct the same eight deterministic training batches and the merger asserts identical sample IDs and ordering.

In [ ]:
import scripts.run_quick_trajectory_t4x2 as quick_runner
quick_runner = importlib.reload(quick_runner)
OUTPUT_DIR = Path('/kaggle/working/rq2-v3-quick-trajectory')
DATASET_ROOT = Path('/kaggle/working/cifar100-data')
started = time.perf_counter()
result = quick_runner.run_quick_trajectory_t4x2(
    HT_ROOT, OUTPUT_DIR, DATASET_ROOT, gpu_ids=[0,1]
)
result['metadata']['git_commit'] = GIT_COMMIT
(OUTPUT_DIR/'metadata.json').write_text(json.dumps(result['metadata'], indent=2)+'\n')
print(f"Diagnostic completed in {(time.perf_counter()-started)/60:.1f} minutes")
print(json.dumps(result['metadata'], indent=2))

## Inspect trajectory variance, total-noise contribution, and oracle drift

In [ ]:
import pandas as pd
from IPython.display import display, Image
for name in ('quick_trajectory_variance.csv','quick_total_variance.csv','quick_policy_distance.csv'):
    print('\n', name); display(pd.read_csv(OUTPUT_DIR/name))
display(pd.read_csv(OUTPUT_DIR/'quick_oracle_drift.csv'))
display(Image(filename=str(OUTPUT_DIR/'quick_trajectory_variance.png')))
display(Image(filename=str(OUTPUT_DIR/'quick_total_variance.png')))
display(Image(filename=str(OUTPUT_DIR/'quick_oracle_drift.png')))

## Validate and export

In [ ]:
required = [
    'quick_trajectory_variance.csv','quick_total_variance.csv',
    'quick_oracle_drift.csv','quick_policy_distance.csv',
    'quick_trajectory_variance.png','quick_total_variance.png','quick_oracle_drift.png',
    'metadata.json','runtime_by_path.csv',
]
missing = [name for name in required if not (OUTPUT_DIR/name).is_file() or (OUTPUT_DIR/name).stat().st_size == 0]
assert not missing, f'Missing diagnostic artifacts: {missing}'
metadata = json.loads((OUTPUT_DIR/'metadata.json').read_text())
assert metadata['training_performed'] is False and metadata['optimizer_steps'] == 0
assert metadata['test_used'] is False and metadata['same_fixed_batch_ids_and_order'] is True
bundle_path = Path('/kaggle/working/rq2-v3-quick-trajectory.zip')
with zipfile.ZipFile(bundle_path, 'w', compression=zipfile.ZIP_DEFLATED, allowZip64=True) as bundle:
    for path in OUTPUT_DIR.rglob('*'):
        if path.is_file(): bundle.write(path, path.relative_to(OUTPUT_DIR))
print('Download/persist:', bundle_path, f'{bundle_path.stat().st_size/2**20:.1f} MiB')
bundle_path